In [1]:
# Install PySCF and GPU4PySCF (CUDA 12.x version for Colab's T4 GPU)
# cuTENSOR and CuPy are also installed for maximum tensor contraction acceleration.
!pip install pyscf gpu4pyscf-cuda12x cupy-cuda12x pyscf-dispersion geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 386.0/386.0 kB 7.6 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.6/51.6 MB 37.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.5/135.5 MB 13.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 159.7/159.7 MB 11.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 82.8 MB/s eta 0:00:00:00:0100:01
  Created wheel for geometric: filename=geometric-1.1-py3-none-any.whl size=402087 sha256=d7fd35c12063874210c1aab63ce7f8f32f925f932a9ae47be1623fbd4bbd970b
  Stored in directory: /root/.cache/pip/wheels/df/1e/9a/763451b0dfd8db47fc02239e33cdf138cbebdbea352bb0871b
Successfully built geometric


**Only use the next cell in case multiple GPUs are used**

In [2]:
# Must be set BEFORE any gpu4pyscf imports
import gpu4pyscf.__config__ as gpu4pyscf_config
gpu4pyscf_config.num_devices = 2

# Verify both GPUs
import cupy as cp
for i in range(2):
    with cp.cuda.Device(i):
        free, total = cp.cuda.runtime.memGetInfo()
        print(f"GPU {i}: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")

/usr/local/lib/python3.12/dist-packages/gpu4pyscf/lib/cutensor.py:154: UserWarning: using cupy as the tensor contraction engine.
  warnings.warn(f'using {contract_engine} as the tensor contraction engine.')


GPU 0: 15.5 GB free / 15.6 GB total
GPU 1: 15.5 GB free / 15.6 GB total


In [3]:
import os
# Please use an xTB/other tool optimised file. Example naming is xtbopt.xyz. Please refrain from using other file formats 
# Copy the file path uploaded as a dataset in the input tab of Kaggle or system path if run locally 
xyz_filename = "/kaggle/input/datasets/pcsciprav/xtbopt-example/compound.xyz"

print(f"Successfully loaded: {xyz_filename}")

Successfully loaded: /kaggle/input/datasets/pcsciprav/xtbopt-example/compound.xyz


In [4]:
import cupy as cp

# Verify both GPUs are visible
n_gpus = cp.cuda.runtime.getDeviceCount()
print(f"GPUs available: {n_gpus}")
for i in range(n_gpus):
    with cp.cuda.Device(i):
        props = cp.cuda.runtime.getDeviceProperties(i)
        print(f"  GPU {i}: {props['name'].decode()}, {props['totalGlobalMem'] / 1e9:.1f} GB")

GPUs available: 2
  GPU 0: Tesla T4, 15.6 GB
  GPU 1: Tesla T4, 15.6 GB


In [5]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0,1'  # Enable both T4s before GPU objects

import pyscf
from pyscf import gto
from gpu4pyscf.dft import uks
from pyscf.geomopt.geometric_solver import optimize
import cupy as cp

# 2. Initialize the Molecule from the uploaded xyz file
with open(xyz_filename, 'r') as f:
    lines = f.readlines()

# XYZ format: line 0 = atom count, line 1 = comment, line 2+ = atoms
n_atoms = int(lines[0].strip())
atom_lines = lines[2:2 + n_atoms]  # skip the xtb comment line
xyz_contents = ''.join(atom_lines)

mol = gto.M(
    atom=xyz_contents,
    basis='def2-SVP', # Use def2-TZVP for higher accuracy, def2-SVP is lighter and causes less OOM issues, use 3-21g for simple checks
    charge=0,
    spin=2,
    verbose=4,
    unit='angstrom'
)
mol.max_memory = 9000
print(f"\nMolecule initialized: {mol.natm} atoms, {mol.nao} atomic orbitals.")

# 3. Configure the GPU-Accelerated DFT Calculation
mf = uks.UKS(mol)
mf.xc = 'r2scan-d3bj'
mf = mf.density_fit()

# Free all VRAM before optimization
cp.get_default_memory_pool().free_all_blocks()
cp.get_default_pinned_memory_pool().free_all_blocks()

# Lower memory usage per SCF cycle
mf.max_memory = 9000
mf.max_cycle = 400
mf.conv_tol = 1e-8
mf.conv_tol_grad = 1e-5
mf.diis_space = 12
mf.level_shift = 0.3
mf.damp = 0.4

# DO NOT use a fresh minao guess here
# mf.init_guess = 'minao'

# 4. RUN GEOMETRY OPTIMIZATION
cp.get_default_memory_pool().free_all_blocks()
cp.get_default_pinned_memory_pool().free_all_blocks()

print("\nStarting GPU-accelerated Geometry Optimization...")

def save_callback(envs):
    mol_envs = envs['mol']
    mol_envs.tofile('/kaggle/working/dft_optimized_checkpoint.xyz')

mol_eq = optimize(
    mf,
    maxsteps=350,
    trust=0.03,
    tmax=0.03,
    callback=save_callback,
    assert_convergence=False
)

# 5. Save the results
output_file = 'dft_optimized.xyz'
mol_eq.tofile(output_file)

print("\n" + "="*50)
print(f"Optimization complete! New coordinates saved to: {output_file}")
print("="*50)

System: uname_result(system='Linux', node='26500c4e035e', release='6.6.113+', version='#1 SMP Sat Jan 17 11:20:45 UTC 2026', machine='x86_64')  Threads 4
Python 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy 2.0.2  scipy 1.16.3  h5py 3.15.1
Date: Tue Mar 31 16:20:48 2026
PySCF version 2.12.1
PySCF path  /usr/local/lib/python3.12/dist-packages/pyscf/__init__.py
CUDA Environment
    CuPy 14.0.1
    CUDA Path None
    CUDA Build Version 12090
    CUDA Driver Version 13000
    CUDA Runtime Version 12090
CUDA toolkit
    cuSolver (11, 7, 3)
    cuBLAS 120804
    cuTENSOR None
Device info
    Device name b'Tesla T4'
    Device global memory 14.56 GB
    CuPy memory fraction 0.9
    Num. Devices 2
GPU4PySCF 1.6.1
GPU4PySCF path  /usr/local/lib/python3.12/dist-packages/gpu4pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 34
[INPUT] num. electrons = 384
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 2
[INPUT] symmetry False subgroup None
[INPUT] Mol

geometric-optimize called with the following command line:
/usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py -f /root/.local/share/jupyter/runtime/kernel-1644f0f4-d859-4e1c-9bd4-65c6cd364137.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%%%%%,  **              ********    


Geometry optimization cycle 1
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   N   0.868520  -0.095920   0.005340   -0.000000  0.000000  0.000000
   N   2.213970  -0.095920   0.005340    0.000000  0.000000  0.000000
   N   2.886690   1.069050   0.028240    0.000000  0.000000 -0.000000
   N   4.232140   1.069050   0.028240    0.000000  0.000000 -0.000000
  Pt   5.387140   3.069180   0.067560    0.000000  0.000000  0.000000
   N   7.157280   2.068540  -1.028540    0.000000  0.000000  0.000000
   O   7.203480   2.068780  -2.397760    0.000000  0.000000  0.000000
   O   8.055520   1.537190  -0.384900    0.000000  0.000000  0.000000
  Pt   3.042920   4.455900  -1.612210   -0.000000  0.000000  0.000000
   N   0.978000   4.377100  -0.579740    0.000000 -0.000000  0.000000
   O   0.846090   3.773330   0.642950    0.000000  0.000000  0.000000
   O   0.000000   4.875650  -1.126000    0.000000  0.000000  0.000000
   N   4.542290   4.150870  -1.6

Step    0 : Gradient = 1.012e+01/4.231e+01 (rms/max) Energy = -7832.5662236780
Hessian Eigenvalues: 2.30000e-02 2.30000e-02 2.30000e-02 ... 5.60016e-01 9.27101e-01 9.27108e-01



Geometry optimization cycle 2
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   N   0.875628  -0.053868   0.033134    0.007108  0.042052  0.027794
   N   2.220842  -0.065540   0.026148    0.006872  0.030380  0.020808
   N   2.903144   1.093336   0.038906    0.016454  0.024286  0.010666
   N   4.247042   1.082696   0.031807    0.014902  0.013646  0.003567
  Pt   5.394401   3.077034   0.053955    0.007261  0.007854 -0.013605
   N   7.153113   2.076785  -1.042702   -0.004167  0.008245 -0.014162
   O   7.190438   2.070143  -2.410666   -0.013042  0.001363 -0.012906
   O   8.054280   1.551233  -0.402025   -0.001240  0.014043 -0.017125
  Pt   3.049923   4.447094  -1.621219    0.007003 -0.008806 -0.009009
   N   0.990670   4.373530  -0.584090    0.012670 -0.003570 -0.004350
   O   0.864774   3.777348   0.642203    0.018684  0.004018 -0.000747
   O   0.011241   4.867461  -1.130526    0.011241 -0.008189 -0.004526
   N   4.537035   4.145178  -1.6

Step    1 : Displace = 2.999e-02/6.464e-02 (rms/max) Trust = 3.000e-02 (=) Grad = 1.033e+01/4.320e+01 (rms/max) E (change) = -7835.1737531751 (-2.608e+00) Quality = 1.010
Eigenvalues below 1.0000e-05 (-8.7995e+00) - returning guess
Hessian Eigenvalues: 2.30000e-02 2.30000e-02 2.30000e-02 ... 5.60016e-01 9.27101e-01 9.27108e-01



Geometry optimization cycle 3
Cartesian coordinates (Angstrom)
 Atom        New coordinates             dX        dY        dZ
   N   0.882983  -0.011736   0.060687    0.007355  0.042132  0.027553
   N   2.227833  -0.035097   0.046901    0.006991  0.030443  0.020753
   N   2.919597   1.117572   0.049419    0.016453  0.024236  0.010512
   N   4.261887   1.096287   0.035407    0.014846  0.013591  0.003600
  Pt   5.401695   3.085024   0.040402    0.007294  0.007990 -0.013553
   N   7.149334   2.084966  -1.056718   -0.003779  0.008181 -0.014015
   O   7.177856   2.071338  -2.423392   -0.012583  0.001194 -0.012726
   O   8.053455   1.565239  -0.418929   -0.000825  0.014006 -0.016904
  Pt   3.056674   4.438333  -1.630187    0.006751 -0.008760 -0.008968
   N   1.003000   4.369888  -0.588168    0.012330 -0.003642 -0.004078
   O   0.883248   3.781343   0.641725    0.018474  0.003995 -0.000478
   O   0.022015   4.859113  -1.134676    0.010774 -0.008348 -0.004149
   N   4.531980   4.139599  -1.6

KeyboardInterrupt: 

In [6]:
import os
for f in os.listdir('/kaggle/working'):
    print(f)
import glob
print(glob.glob('/kaggle/working/*.xyz'))
print(glob.glob('/kaggle/working/**/*.xyz', recursive=True))

dft_optimized_checkpoint.xyz
hessian_matrix.npy
dft_optimized.xyz
.virtual_documents
dft_optimized_final.xyz
['/kaggle/working/dft_optimized_checkpoint.xyz', '/kaggle/working/dft_optimized.xyz', '/kaggle/working/dft_optimized_final.xyz']
['/kaggle/working/dft_optimized_checkpoint.xyz', '/kaggle/working/dft_optimized.xyz', '/kaggle/working/dft_optimized_final.xyz']


In [7]:
# Cell 4: Vibrational Frequencies and Thermochemistry
from gpu4pyscf.dft import rks
from gpu4pyscf.hessian import rks as gpu_hessian
from pyscf.hessian import thermo
import cupy as cp
import numpy as np
from pyscf import gto
import numpy as np

with open('/kaggle/working/dft_optimized_checkpoint.xyz', 'r') as f:
    lines = f.readlines()
n_atoms = int(lines[0].strip())
atom_lines = lines[2:2 + n_atoms]
xyz_contents = ''.join(atom_lines)

mol_eq = gto.M(
    atom = xyz_contents,
    basis = 'def2-svp',
    charge = 0,
    spin = 0,
    verbose = 4,
    unit = 'angstrom'
)
mol_eq.max_memory = 16000

print("1. Setting up new SCF for the optimized geometry...")
mf_eq = rks.RKS(mol_eq)
mf_eq.xc = 'r2scan-d3bj'
mf_eq = mf_eq.density_fit()
mf_eq.conv_tol = 1e-9
e_opt = mf_eq.kernel()

print("\n2. Calculating analytical Hessian (Frequencies) on the GPU...")
hess = gpu_hessian.Hessian(mf_eq)
hess_matrix = hess.kernel()
# === AUTO-FIX IMAGINARY MODES LOOP ===

max_iter = 5
step = 0.1  # distortion magnitude (Å)

for i in range(max_iter):
    print("\n" + "="*40)
    print(f"Iteration {i+1}")
    print("="*40)

    # --- Compute Hessian ---
    hess = gpu_hessian.Hessian(mf_eq)
    hess_matrix = hess.kernel()

    # --- Reshape to 3N x 3N ---
    hess_2d = hess_matrix.reshape(3*n_atoms, 3*n_atoms)

    # --- Eigen decomposition ---
    eigvals, eigvecs = np.linalg.eigh(hess_2d)

    # --- Find REAL negative modes (ignore tiny noise) ---
    neg_indices = np.where(eigvals < -1e-3)[0]

    print(f"Negative eigenvalues: {len(neg_indices)}")
    print("Smallest eigenvalues:", eigvals[:6])

    # --- Stop if clean ---
    if len(neg_indices) == 0:
        print("✅ Reached true minimum (no real imaginary modes)")
        break

    # --- Pick most negative mode ---
    mode_idx = neg_indices[0]
    mode = eigvecs[:, mode_idx].reshape(n_atoms, 3)

    coords = mol_eq.atom_coords()

    # --- Try BOTH directions ---
    best_energy = None
    best_coords = None

    for direction in [+1, -1]:
        trial_coords = coords + direction * step * mode

        mol_trial = mol_eq.copy()
        mol_trial.set_geom_(trial_coords, unit='angstrom')

        mf_trial = rks.RKS(mol_trial)
        mf_trial.xc = 'r2scan-d3bj'
        mf_trial = mf_trial.density_fit()
        mf_trial.conv_tol = 1e-9

        e_trial = mf_trial.kernel()

        print(f"  Direction {direction:+} Energy: {e_trial:.6f}")

        if best_energy is None or e_trial < best_energy:
            best_energy = e_trial
            best_coords = trial_coords

    # --- Update geometry to best ---
    mol_eq.set_geom_(best_coords, unit='angstrom')

    # --- Re-optimise ---
    print("Re-optimising...")
    mf_eq = rks.RKS(mol_eq)
    mf_eq.xc = 'r2scan-d3bj'
    mf_eq = mf_eq.density_fit()
    mf_eq.conv_tol = 1e-9
    mf_eq.kernel()

print("\nLoop complete.")
print("\n3. Calculating Thermochemistry (298.15 K, 1 atm)...")
freq_info = thermo.harmonic_analysis(mol_eq, hess_matrix)
thermo_data = thermo.thermo(mf_eq, freq_info['freq_au'], 298.15, 101325.0)

print("\n--- Thermochemistry Summary ---")
thermo.dump_thermo(mol_eq, thermo_data)

# MEMORY MANAGEMENT & BACKUP
print("\n4. Saving data to disk and purging VRAM...")

# Step A: Save the massive Hessian matrix to a file so we don't lose it
np.save('hessian_matrix.npy', np.asarray(hess_matrix))
print("   -> Hessian securely saved to 'hessian_matrix.npy'")

# Step B: Explicitly delete the giant objects from Python's memory
del hess
del freq_info
del thermo_data

# Step C: Force the GPU to empty the trash now that Python let go of the variables
cp.get_default_memory_pool().free_all_blocks()
print("   -> GPU VRAM successfully flushed!")

print("\n" + "="*50)
print("FREQUENCY & THERMO CALCULATION COMPLETE")
print("="*50)

System: uname_result(system='Linux', node='26500c4e035e', release='6.6.113+', version='#1 SMP Sat Jan 17 11:20:45 UTC 2026', machine='x86_64')  Threads 4
Python 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
numpy 2.0.2  scipy 1.16.3  h5py 3.15.1
Date: Tue Mar 31 17:15:52 2026
PySCF version 2.12.1
PySCF path  /usr/local/lib/python3.12/dist-packages/pyscf/__init__.py
CUDA Environment
    CuPy 14.0.1
    CUDA Path None
    CUDA Build Version 12090
    CUDA Driver Version 13000
    CUDA Runtime Version 12090
CUDA toolkit
    cuSolver (11, 7, 3)
    cuBLAS 120804
    cuTENSOR None
Device info
    Device name b'Tesla T4'
    Device global memory 14.56 GB
    CuPy memory fraction 0.9
    Num. Devices 2
GPU4PySCF 1.6.1
GPU4PySCF path  /usr/local/lib/python3.12/dist-packages/gpu4pyscf

[CONFIG] conf_file None
[INPUT] verbose = 4
[INPUT] num. atoms = 34
[INPUT] num. electrons = 384
[INPUT] charge = 0
[INPUT] spin (= nelec alpha-beta = 2S) = 0
[INPUT] symmetry False subgroup None
[INPUT] Mol

KeyboardInterrupt: 

In [ ]:
import numpy as np
import cupy as cp

print("="*50)
print("HESSIAN / EIGENVALUE SUMMARY")
print("="*50)

H = hess_matrix
if isinstance(H, cp.ndarray):
    H = cp.asnumpy(H)

# block Hessian (natm,natm,3,3) -> full matrix
natm = H.shape[0]
Hfull = H.transpose(0,2,1,3).reshape(3*natm, 3*natm)

w = np.linalg.eigvalsh(Hfull)

print("Shape of full Hessian:", Hfull.shape)
print("Smallest 20 eigenvalues:")
print(w[:20])
print("Largest 10 eigenvalues:")
print(w[-10:])

tol = 1e-8
nneg = np.sum(w < -tol)
nzero = np.sum(np.abs(w) <= tol)
npos = np.sum(w > tol)

print("Negative eigenvalues:", nneg)
print("Near-zero eigenvalues:", nzero)
print("Positive eigenvalues:", npos)
print("="*50)
print("SOFTEST MODE ATOMIC PARTICIPATION")
print("="*50)

eigvals, eigvecs = np.linalg.eigh(Hfull)

for mode in range(min(5, len(eigvals))):
    vec = eigvecs[:, mode].reshape(natm, 3)
    atom_amp = np.linalg.norm(vec, axis=1)
    top = np.argsort(atom_amp)[::-1][:10]
    print(f"\nMode {mode}  eigenvalue = {eigvals[mode]: .8f}")
    for i in top:
        print(f"Atom {i:2d} {mol_eq.atom_symbol(i):2s}  amplitude = {atom_amp[i]:.6f}")
del hess_matrix

In [ ]:
# Cell 5: Excited States (TD-DFT) for UV-Vis
from gpu4pyscf.tdscf import rks as gpu_tdscf
import cupy as cp
from pyscf import gto
from gpu4pyscf.dft import rks

with open('dft_optimized_checkpoint.xyz', 'r') as f:
    lines = f.readlines()
n_atoms = int(lines[0].strip())
atom_lines = lines[2:2 + n_atoms]
xyz_contents = ''.join(atom_lines)

mol_eq = gto.M(
    atom = xyz_contents,
    basis = 'def2-svp',
    charge = 0,
    spin = 0,
    verbose = 4,
    unit = 'angstrom'
)
mol_eq.max_memory = 10000

mf_eq = rks.RKS(mol_eq)
mf_eq.xc = 'r2scan-d3bj'
mf_eq = mf_eq.density_fit()
mf_eq.conv_tol = 1e-9
mf_eq.kernel()
print("Cleaning up GPU VRAM from previous calculations...")
# ==========================================
# CRITICAL VRAM CLEANUP
# Forces the GPU to release cached memory from the Hessian calc
# ==========================================
cp.get_default_memory_pool().free_all_blocks()

print("\nStarting GPU-accelerated TD-DFT...")
print("Calculating the first 5 singlet excited states...\n")

# Initialize Time-Dependent DFT using our optimized molecule's SCF object (mf_eq)
td = gpu_tdscf.TDDFT(mf_eq)
td.nstates = 5       # Number of excited states to calculate
td.singlet = True    # Look for singlet-singlet transitions

# ==========================================
# VRAM OPTIMIZATION FOR TD-DFT
# Restricts the Davidson solver's maximum memory footprint
# ==========================================
td.max_space = 15

# Run the calculation
td.kernel()

print("\n" + "="*50)
print("TD-DFT EXCITATION ENERGIES:")
print("="*50)
# This will print the transition energies (in eV), wavelengths (in nm), and oscillator strengths (intensity)
td.analyze()

In [ ]:
print("="*50)
print("TDDFT SUMMARY")
print("="*50)

print("Excitation energies (eV):")
print(cp.asnumpy(td.e) * 27.211386245988 if hasattr(td.e, "get") or "cupy" in str(type(td.e)).lower() else td.e * 27.211386245988)

print("\nOscillator strengths:")
try:
    print(td.oscillator_strength())
except Exception as e:
    print("Could not get oscillator strengths directly:", e)

In [ ]:
import cupy as cp
from pyscf.scf import hf

print("="*50)
print("1. MOLECULAR DIPOLE MOMENT")
print("="*50)
dipole = mf_eq.dip_moment()

print("\n" + "="*50)
print("2. MULLIKEN ATOMIC CHARGES")
print("="*50)

dm = mf_eq.make_rdm1()
s = mol_eq.intor('int1e_ovlp')

# Convert GPU array to CPU NumPy array
dm = cp.asnumpy(dm)

mulliken_pop, mulliken_chg = hf.mulliken_pop(mol_eq, dm, s=s)

print("\nAtom | Charge")
print("-" * 20)
for i, charge in enumerate(mulliken_chg):
    atom_symbol = mol_eq.atom_symbol(i)
    print(f"{i:2d} {atom_symbol:2s} | {charge: .4f}")

In [ ]:
print("="*50)
print("ATOM INDEX MAP")
print("="*50)

for i in range(mol_eq.natm):
    x, y, z = mol_eq.atom_coord(i)
    print(f"{i:2d} {mol_eq.atom_symbol(i):2s}  {x: .6f} {y: .6f} {z: .6f}")

In [ ]:
import numpy as np

print("="*50)
print("CHARGE SUMMARY")
print("="*50)

charges = np.array(mulliken_chg)

print("Min charge:", charges.min())
print("Max charge:", charges.max())
print("Mean charge:", charges.mean())
print("Std dev:", charges.std())

print("\nTop 10 most positive atoms")
for i in np.argsort(charges)[::-1][:10]:
    print(f"Atom {i:2d} {mol_eq.atom_symbol(i):2s}  q = {charges[i]: .6f}")

print("\nTop 10 most negative atoms")
for i in np.argsort(charges)[:10]:
    print(f"Atom {i:2d} {mol_eq.atom_symbol(i):2s}  q = {charges[i]: .6f}")

In [ ]:
# Cell 7: Save the optimized coordinates
output_xyz = 'dft_optimized_final.xyz'
mol_eq.tofile(output_xyz)
print(f"Saved to /kaggle/working/{output_xyz}")
print("Download it from the Output tab on the right panel.")